# Personalized FL for Heart Disease Risk Prediction — Colab Runner

Runs this project's experiments in Google Colab instead of the local Windows machine, which blocks several native ML library DLLs under Smart App Control (see `README.md`, "Environment note" section).

**Usage:** Runtime → Change runtime type → CPU is fine (no GPU needed; this project targets standard hardware). Then Runtime → Run all.

This notebook: (1) clones the repo, (2) installs `requirements.txt` fresh, (3) re-verifies Phase 3/4/5 still reproduce their committed results, (4) runs Phase 6's SHAP explainability analysis.

## 1. Clone the repository

In [ ]:
REPO_URL = "https://github.com/Teena2812/PBL-5.git"

import os
if os.path.exists("project"):
    %cd project
    !git pull
else:
    !git clone $REPO_URL project
    %cd project

## 2. Install dependencies

Colab already ships numpy/pandas/scikit-learn/matplotlib/torch, but we install
from `requirements.txt` to match the pinned versions used for the committed
results (pandas==2.2.3 and matplotlib==3.8.4 were pinned only to work around
the LOCAL machine's Windows Application Control block -- those specific pins
aren't needed on Colab, but installing them keeps results reproducible
against what's documented in the README).

In [ ]:
!pip install -q -r requirements.txt

## 3. Sanity check: core libraries import cleanly

In [ ]:
import numpy, pandas, sklearn, matplotlib, torch, flwr
print("numpy", numpy.__version__)
print("pandas", pandas.__version__)
print("scikit-learn", sklearn.__version__)
print("matplotlib", matplotlib.__version__)
print("torch", torch.__version__)
print("flwr", flwr.__version__)

from sklearn.ensemble import RandomForestClassifier
import numpy as np
RandomForestClassifier(n_estimators=5).fit(np.random.rand(10, 3), np.random.randint(0, 2, 10))
print("RandomForestClassifier: OK")

## 4. Re-verify Phase 3 (Local ML / Centralized ML baselines)

Should reproduce the committed results: Local ML mean accuracy ~0.833 (RF) / ~0.819 (LR), Centralized ML ~0.842 (RF) / ~0.855 (LR).

In [ ]:
!python experiments/phase3_baselines.py

## 5. Re-verify Phase 4 (FedAvg via Flower)

Should reproduce: FedAvg final global weighted accuracy ~0.829 (single seed=42 run).

In [ ]:
!python experiments/phase4_fedavg.py

In [ ]:
!python experiments/phase4_nn_baselines.py

### Optional: Phase 4/5 multi-seed robustness checks (slower, ~5-15 min each)

Only re-run these if you need to re-verify the multi-seed numbers already committed in `experiments/results/phase4_multiseed_*.csv` and `phase5_multiseed_*.csv` -- otherwise skip to Phase 6 below.

In [ ]:
# !python experiments/phase4_multiseed_comparison.py
# !python experiments/phase5_fedprox.py
# !python experiments/phase5_multiseed_comparison.py
# !python experiments/phase5_equity_analysis.py

## 6. Phase 6: SHAP explainability

This is the analysis that Smart App Control blocked locally (scikit-learn's
compiled tree extensions, then numba, then more sklearn submodules --
whack-a-mole DLL blocking). Should run cleanly on Colab.

In [ ]:
import shap
print("shap", shap.__version__, "-- import OK")

In [ ]:
!python experiments/phase6_shap_analysis.py

## 7. Display the generated charts inline

In [ ]:
from IPython.display import Image, display
import glob

for path in sorted(glob.glob("experiments/results/phase6_*.png")):
    print(path)
    display(Image(filename=path))

## 9. Phase 7 prep: save personalized model checkpoints + preprocessing artifact

Saves each hospital's final personalized model (FedProx + fine-tuning, same
pipeline as Phase 5) as a PyTorch state_dict, plus a `preprocessing.json`
capturing the standardization constants and categorical encoding needed to
transform a brand-new raw patient record at inference time. These back
Phase 7's live single-patient prediction endpoint (inference + single-instance
SHAP only -- no retraining happens there).

The printed personalized metrics should exactly match Phase 5's
`[Personalized FL]` results above, as a sanity check.

In [ ]:
!python experiments/save_personalized_models.py

In [ ]:
!zip -r phase7_models.zip experiments/results/models/
from google.colab import files
files.download("phase7_models.zip")

## 8. Download results back to sync with the local repo

Zips `experiments/results/` so you can download it and copy the new Phase 6
outputs (CSVs + PNGs) back into the local checkout before committing.

In [ ]:
!zip -r phase6_results.zip experiments/results/phase6_*
from google.colab import files
files.download("phase6_results.zip")